In [1]:
%reset -f

In [2]:
%reset -f

In [3]:
%reset -f

In [4]:
# stop 

In [5]:
!nvidia-smi

Tue Oct 20 10:48:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.97                 Driver Version: 595.97         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   43C    P8              3W /   95W |    1064MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# 1) Setup

In [6]:
import os 
os.chdir('D:/College/AI/LLM Learning/Module 0/Module-0-Transformers-from-Scratch')

os.getcwd()

'D:\\College\\AI\\LLM Learning\\Module 0\\Module-0-Transformers-from-Scratch'

In [7]:
import random
import numpy as np

seed = 42
random.seed(seed)
np.random.seed(seed)

# 2) Packages

In [ ]:
# =====================================================================
# 1. Standard Library & Utilities
# =====================================================================
import os
import sys
import math
import time
import json
import random
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
from collections import Counter
from datasets import load_dataset, Dataset

In [9]:
# =====================================================================
# 2. General Data Manipulation, Science & Visualization
# =====================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [ ]:
# =====================================================================
# 3. Deep Learning & PyTorch Core
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR, CosineAnnealingLR
from typing import Optional

In [11]:
# =====================================================================
# 4. Tokenization & NLP Metrics
# =====================================================================
# Custom subword tokenization from scratch (no pretrained weights)
# If using Hugging Face's tokenizers library:
try:
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
except ImportError:
    pass

# Metric computation (ROUGE scores for abstractive summarization)
try:
    from rouge_score import rouge_scorer
except ImportError:
    pass

In [12]:
# =====================================================================
# 5. Device Verification & Seeding Utility
# =====================================================================
def set_seed(seed) -> None:
    # random.seed(seed)
    # np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed(seed=seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)} GB")

# Hardware / Device Status
print("-" * 50)
try:
    cuda_available = torch.cuda.is_available()
    device_name = torch.cuda.get_device_name(0) if cuda_available else "CPU only"
    print(f"CUDA Available: {cuda_available}")
    print(f"Active Device:  {device_name}")
except ImportError:
    print("PyTorch not installed.")
print("-" * 50)

Using device: cuda
Device Name: NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM: 8.0 GB
--------------------------------------------------
CUDA Available: True
Active Device:  NVIDIA GeForce RTX 4060 Laptop GPU
--------------------------------------------------


# 3) Load Dataset

In [13]:
import os
import pandas as pd
from tokenizers import Tokenizer

In [14]:
# =====================================================================
# 3. Load Processed Datasets & Custom Tokenizer
# =====================================================================

In [15]:
# Define paths to saved processed artifacts
PROCESSED_DIR = os.path.join(os.getcwd(), "data", "processed")
TOKENIZER_PATH = os.path.join(PROCESSED_DIR, "tokenizer.json")

In [16]:
# 1. Load the trained Tokenizer
if not os.path.exists(TOKENIZER_PATH):
    raise FileNotFoundError(f"Tokenizer not found at {TOKENIZER_PATH}. Run text_processing.ipynb first!")

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer

Tokenizer(version="1.0", truncation=None, padding=None, added_tokens=[{"id":0, "content":"<PAD>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":1, "content":"<UNK>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":2, "content":"<SOS>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}, {"id":3, "content":"<EOS>", "single_word":False, "lstrip":False, "rstrip":False, "normalized":False, "special":True}], normalizer=NFKC(), pre_tokenizer=Whitespace(), post_processor=None, decoder=WordPiece(prefix="##", cleanup=True), model=WordPiece(unk_token="<UNK>", continuing_subword_prefix="##", max_input_chars_per_word=100, vocab={"<PAD>":0, "<UNK>":1, "<SOS>":2, "<EOS>":3, "!":4, """:5, "#":6, "$":7, "%":8, "&":9, "'":10, "(":11, ")":12, "*":13, "+":14, ",":15, "-":16, ".":17, "/":18, "0":19, "1":20, "2":21, "3":22, "4":23, "5":24, "6":25, "7":26, "8":27, "9":28, "

In [17]:
# Extract Special Token IDs & Vocabulary Size
PAD_IDX = tokenizer.token_to_id("<PAD>")
UNK_IDX = tokenizer.token_to_id("<UNK>")
SOS_IDX = tokenizer.token_to_id("<SOS>")
EOS_IDX = tokenizer.token_to_id("<EOS>")
VOCAB_SIZE = tokenizer.get_vocab_size()

print(f"Vocabulary Size : {VOCAB_SIZE:,}")
print(f"Special Tokens  : <PAD>={PAD_IDX}, <UNK>={UNK_IDX}, <SOS>={SOS_IDX}, <EOS>={EOS_IDX}")

Vocabulary Size : 8,000
Special Tokens  : <PAD>=0, <UNK>=1, <SOS>=2, <EOS>=3


In [18]:
# 2. Load Processed Datasets
train_df = pd.read_json(os.path.join(PROCESSED_DIR, "train.json"))
val_df   = pd.read_json(os.path.join(PROCESSED_DIR, "val.json"))
test_df  = pd.read_json(os.path.join(PROCESSED_DIR, "test.json"))

print(f"Train samples   : {len(train_df):,}")
print(f"Val samples     : {len(val_df):,}")
print(f"Test samples    : {len(test_df):,}")

train_df

Train samples   : 14,685
Val samples     : 818
Test samples    : 814


,id,article,summary
0,13818513,Amanda: I baked cookies. Do you want some?\nJe...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\nKim: Bad mood tbh, I was ...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\nSam: i ...,"Sam is confused, because he overheard Rick com..."
...,...,...,...
14680,13863028,Romeo: You are on my ‘People you may know’ lis...,Romeo is trying to get Greta to add him to her...
14681,13828570,Theresa: <file_photo>\nTheresa: <file_photo>\n...,Theresa is at work. She gets free food and fre...
14682,13819050,John: Every day some bad news. Japan will hunt...,Japan is going to hunt whales again. Island an...
14683,13828395,Jennifer: Dear Celia! How are you doing?\nJenn...,Celia couldn't make it to the afternoon with t...


In [19]:
sample_text = train_df.iloc[0]["summary"]
encoded = tokenizer.encode(sample_text)
decoded = tokenizer.decode(encoded.ids)

print(f"Original Text: {sample_text}")
print(f"Token IDs    : {encoded.ids[:10]}... (Total {len(encoded.ids)} tokens)")
print(f"Decoded Text : {decoded}")

Original Text: Amanda baked cookies and will bring Jerry some tomorrow.
Token IDs    : [2064, 7872, 4459, 990, 1019, 1584, 2502, 1099, 1253, 17]... (Total 10 tokens)
Decoded Text : Amanda baked cookies and will bring Jerry some tomorrow.


# 6) DataLoader Class

In [20]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Any

In [21]:
class data_loader (Dataset):

    """
    Custom PyTorch Dataset for Sequence-to-Sequence Abstractive Summarization.
    Encodes text using the custom WordPiece tokenizer and formats inputs
    for teacher-forced Transformer training.
    """

    def __init__(self, 
                df : pd.DataFrame, 
                tokenizer, 
                max_src_len : int = 128, 
                max_tgt_len : int = 64) :
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        
        # Cache special tokens
        self.pad_idx = tokenizer.token_to_id("<PAD>")
        self.sos_idx = tokenizer.token_to_id("<SOS>")
        self.eos_idx = tokenizer.token_to_id("<EOS>")
        
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx : int) -> Dict[str, List[int]]:

    # ====================================================================================================

        # 1) for each i till dataset length 
        #     Takes Raw DataFrame Row (ex idx i = 0)
        #     article = "Amanda: I baked cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)"
        #     summary = "Amanda baked cookies and will bring Jerry some tomorrow."
        # 
        # 2) Process Input + Output Data
        # 
        #     Step A: Tokenizing the Article (Encoder Input) 
        #         Convert words to tokens from vocab for encoder text (full <article>)
        #         src_encoded = tokenizer.encode(article).ids
        #         Text :   "Amanda: I baked cookies. Do you want some?\nJerry: Sure!\nAmanda: I'll bring you tomorrow :-)"
        #         Tokens : ['Amanda', ':', 'I', 'baked', 'cookies', '.', 'Do', 'you', ..., ')']
        #         Result : [  2064,    25,  97,   41,       7872,    80,  17,    98,  ..., 64 ]
        # 
        #     Step B: Tokenizing the Summary (Decoder Input) 
        #         Convert words to tokens from vocab for decoder text (full <summary>)
        #         tgt_encoded = tokenizer.encode(summary).ids
        #         summary : "Amanda baked cookies and will bring Jerry some tomorrow."
        #         Tokens :  ['Amanda', 'baked', 'cookies', 'and', 'will', 'bring', 'Jerry', 'some', 'tomorrow', '.']
        #         IDs :     [  2064,    7872,     4459,     990,   1019,   1584,     2502,   1099,     1253,     17]
        # 
        #     Step C: Teacher-Forcing Shift
        #         To train the Transformer decoder to predict the next token, we create two versions of the summary:
        #         1) decoder_input : Prepend (Start of Sentence, ID 2)
        #             Input to Decoder = [(SOS), Amanda, baked,..., tomorrow,.]
        #         2) decoder_target: Append (End of Sentence, ID 3) 
        #             Ground Truth Labels = [Amanda, baked,..., tomorrow, ., (EOS)]
        # 
        #         +----------+---------------------------+--------------------------------------------+
        #         | Timestep | Decoder Input (Context)   | Decoder Target (Supervision Label)         |
        #         +----------+---------------------------+--------------------------------------------+
        #         | t_1      | <SOS>    (ID: 2)          | Amanda     (ID: 2064)                      |
        #         | t_2      | Amanda   (ID: 2064)       | baked      (ID: 7872)                      |
        #         | t_3      | baked    (ID: 7872)       | cookies    (ID: 4459)                      |
        #         | ...      | ...                       | ...                                        |
        #         | t_10     | tomorrow (ID: 1253)       | .          (ID: 17)                        |
        #         | t_11     | .        (ID: 17)         | <EOS>      (ID: 3) [Sequence termination]  |
        #         +----------+---------------------------+--------------------------------------------+
        # 
        # 3. Returned Output (Dictionary) 
        #     {
        #         # 1. Encoder Input (Source dialogue tokens):
        #         "encoder_input": [2064, 25, 41, 7872, 4459, 17, 98, 1024, 1102, 1077, 27, ...]
        #         # 2. Decoder Input (Shifted summary starting with <SOS>):
        #         "decoder_input": [2, 2064, 7872, 4459, 990, 1019, 1584, 2502, 1099, 1253, 17]
        #         # 3. Decoder Output / labels for Loss Func (Shifted summary ending with <EOS>):
        #         "decoder_target": [2064, 7872, 4459, 990, 1019, 1584, 2502, 1099, 1253, 17, 3]
        #     }
        # 
        # 4. Note :   We return raw lists instead of padded PyTorch tensors here 
        #             because individual examples have different lengths 
        #             (e.g., summary 1 has 11 tokens, summary 2 has 25 tokens). 
        #             By leaving them as simple lists in __getitem__, the collate_fn 
        #             can look at the 16 examples in a batch, find the single longest one, 
        #             and pad only up to that length, saving lots of GPU memory
        # 
        #             We can always do - 
        #             1) Fixed Padding (every batch is strictly [16, 128] and [16, 64]) 
        #             2) Dynamic Padding (pads each batch only to the max length needed in that batch) 
        #             And Dynamic padding is generally faster for smaller batches + saves memory

    # ====================================================================================================
        
        row = self.df.iloc[idx]
        article_text = str(row["article"])
        summary_text = str(row["summary"])
        
        # 1. Encode Source (Article)
        src_encoded = self.tokenizer.encode(article_text).ids
        # Truncate source sequence
        src_encoded = src_encoded[:self.max_src_len]
        
        # 2. Encode Target (Summary)
        # Leave room for <SOS> and <EOS>
        tgt_encoded = self.tokenizer.encode(summary_text).ids
        tgt_encoded = tgt_encoded[:self.max_tgt_len - 1]
        
        # Teacher-Forcing Sequence Alignment:
        # decoder_input:  [<SOS>, t1, t2, ..., tn]
        # decoder_target: [t1,    t2, ..., tn, <EOS>]
        decoder_input = [self.sos_idx] + tgt_encoded
        decoder_target = tgt_encoded + [self.eos_idx]
        
        return {
            "encoder_input": src_encoded,
            "decoder_input": decoder_input,
            "decoder_target": decoder_target
        }
    
    @staticmethod
    def collate_fn(batch : List[Dict[str, List[int]]], pad_idx : int = 0) -> Dict[str, torch.Tensor]:
        """
        Dynamically pads sequences in the batch to the maximum length of that batch.
        Also constructs source padding masks and causal target masks.
        """
        # Find maximum length in current batch
        max_src_len = max(len(item["encoder_input"]) for item in batch)
        max_tgt_len = max(len(item["decoder_input"]) for item in batch)
        
        batch_size = len(batch)
        
        # Allocate tensors initialized with PAD_IDX
        encoder_input = torch.full((batch_size, max_src_len), pad_idx, dtype=torch.long)
        decoder_input = torch.full((batch_size, max_tgt_len), pad_idx, dtype=torch.long)
        decoder_target = torch.full((batch_size, max_tgt_len), pad_idx, dtype=torch.long)
        
        for i, item in enumerate(batch):
            src_len = len(item["encoder_input"])
            tgt_len = len(item["decoder_input"])
            
            encoder_input[i, :src_len] = torch.tensor(item["encoder_input"], dtype=torch.long)
            decoder_input[i, :tgt_len] = torch.tensor(item["decoder_input"], dtype=torch.long)
            decoder_target[i, :tgt_len] = torch.tensor(item["decoder_target"], dtype=torch.long)
            
        # =====================================================================
        # Attention Masks - Explanation in Following Cells
        # =====================================================================
        # 1. Source Padding Mask: True (or 1) for valid tokens, False (or 0) for PAD tokens
        # Shape: [batch_size, 1, 1, max_src_len] (broadcastable over heads & tgt_len)
        src_mask = (encoder_input != pad_idx).unsqueeze(1).unsqueeze(2)
        
        # 2. Target Padding Mask: [batch_size, 1, 1, max_tgt_len]
        tgt_pad_mask = (decoder_input != pad_idx).unsqueeze(1).unsqueeze(2)
        
        # 3. Causal / Look-Ahead Mask: [1, 1, max_tgt_len, max_tgt_len]
        # Position i can only attend to positions <= i
        # torch.tril basically make the upper-right triangular matrix of 0s where we only have the prev elements with token numbers rest 0
        causal_mask = torch.tril(torch.ones((max_tgt_len, max_tgt_len), dtype=torch.bool)).unsqueeze(0).unsqueeze(1)
        
        # Combined Target Mask: must be both valid token AND not in future
        tgt_mask = tgt_pad_mask & causal_mask
        
        return {
            "encoder_input": encoder_input,      # [B, max_src_len]
            "decoder_input": decoder_input,      # [B, max_tgt_len]
            "decoder_target": decoder_target,    # [B, max_tgt_len]
            "src_mask": src_mask,                # [B, 1, 1, max_src_len]
            "tgt_mask": tgt_mask                 # [B, 1, max_tgt_len, max_tgt_len]
        }

##### Understanding Masks in Attention

In [22]:
# ====================================================================================================
# STEP-BY-STEP BREAKDOWN: HOW ATTENTION MASKS WORK IN TRANSFORMERS
# ====================================================================================================

# In Transformer attention, the core operation calculates an attention score matrix of shape:

#       Shape: [Batch_Size, Num_Heads, Q_len, K_len]

# Where entry (i, j) represents: "How much should Query token i pay attention to Key token j?"

# To prevent the model from learning bad shortcuts, we mask out invalid connections before softmax
# by setting their attention scores to -inf (or -1e9), so their attention weight becomes 0.

# ----------------------------------------------------------------------------------------------------
# 1. SOURCE PADDING MASK (src_mask)
# ----------------------------------------------------------------------------------------------------

#         Line: src_mask = (encoder_input != pad_idx).unsqueeze(1).unsqueeze(2)

#         REAL-LIFE CONCRETE EXAMPLE:
#         Suppose Batch Size = 2, max_src_len = 4, pad_idx = 0.

#         Sentence 1: "I love cats"           -> 3 real tokens + 1 <PAD>
#         Sentence 2: "Dogs rule"             -> 2 real tokens + 2 <PAD>s

#         In numerical token IDs, encoder_input is a 2D tensor of Shape [Batch_Size=2, max_src_len=4]:

#         encoder_input = [
#             [ 42, 108, 999,   0 ],   # Sentence 1: ["I", "love", "cats", "<PAD>"]
#             [ 55, 301,   0,   0 ]    # Sentence 2: ["Dogs", "rule", "<PAD>", "<PAD>"]
#         ]

#         ----------------------------------------------------------------------------------------------------
#         STEP A: The Boolean Condition (encoder_input != pad_idx)
#         ----------------------------------------------------------------------------------------------------
#         Checks every ID: True if real word, False if <PAD> (0).

#         mask_2d = (encoder_input != 0)
#         Shape: [2, 4]   (Batch_Size=2, max_src_len=4)

#         mask_2d = [
#             [ True,  True,  True, False ],   # Sentence 1
#             [ True,  True, False, False ]    # Sentence 2
#         ]

#         ----------------------------------------------------------------------------------------------------
#         STEP B: The First .unsqueeze(1)  (Injects Dimension 1: Attention Heads)
#         ----------------------------------------------------------------------------------------------------
#         In Transformer Attention, we split embeddings into Multiple Heads (e.g., num_heads = 4 or 8).
#         The attention score tensor has 4 dimensions: [Batch, Heads, Query_Len, Key_Len].

#         Calling .unsqueeze(1) inserts a new dimension of size 1 at index 1:

#         mask_3d = mask_2d.unsqueeze(1)
#         Shape: [2, 1, 4]   (Batch_Size=2, Dummy_Head=1, max_src_len=4)

#         mask_3d = [
#             [ [ True,  True,  True, False ] ],   # Sentence 1: 1 list of 4 elements
#             [ [ True,  True, False, False ] ]    # Sentence 2: 1 list of 4 elements
#         ]

#         ----------------------------------------------------------------------------------------------------
#         STEP C: The Second .unsqueeze(2)  (Injects Dimension 2: Query Length)
#         ----------------------------------------------------------------------------------------------------
#         Calling .unsqueeze(2) inserts another dimension of size 1 at index 2:

#         src_mask = mask_3d.unsqueeze(2)
#         Shape: [2, 1, 1, 4]   (Batch_Size=2, Dummy_Head=1, Dummy_Query=1, Key_Len=4)

#         src_mask = [
#             [ [ [ True,  True,  True, False ] ] ],  # Sentence 1
#             [ [ [ True,  True, False, False ] ] ]   # Sentence 2
#         ]

#         ----------------------------------------------------------------------------------------------------
#         STEP D: HOW PYTORCH BROADCASTING MAGICALLY COMBINES THIS IN ATTENTION:
#         ----------------------------------------------------------------------------------------------------
#         In Attention, we calculate:
#         scores = (Q @ K.T) / sqrt(d_k)

#         Suppose:
#         • Batch Size = 2
#         • Num Heads  = 4
#         • Query Len  = 3  (e.g., 3 words asking questions)
#         • Key Len    = 4  (the 4 source words being attended to)

#         Therefore:
#         scores.shape   = [2, 4, 3, 4]
#         src_mask.shape = [2, 1, 1, 4]

#         Notice how PyTorch compares shapes from right to left:
#         scores   : [ 2 ,   4   ,   3   ,   4   ]
#         src_mask : [ 2 ,   1   ,   1   ,   4   ]
#                     ==============================
#         Result   : [ 2 , MATCH , MATCH , MATCH ]

#         The '1' at Heads automatically copies itself 4 times (for all 4 attention heads).
#         The '1' at Query automatically copies itself 3 times (for all 3 query words).

#         For Sentence 1, EVERY query word in EVERY attention head sees:
#             Key:   "I"    "love"  "cats"   <PAD>
#             Mask: [ True,  True,   True,   False ]

#         And when applied:
#         scores.masked_fill(src_mask == False, -1e9)

#         Column 4 (the <PAD> column) becomes -1e9.
#         When softmax runs, e^(-1e9) ≈ 0.00000!
#         Thus, NO query word ever absorbs information from <PAD>!

# ----------------------------------------------------------------------------------------------------
# 2. TARGET PADDING MASK (tgt_pad_mask)
# ----------------------------------------------------------------------------------------------------
#         Line: tgt_pad_mask = (decoder_input != pad_idx).unsqueeze(1).unsqueeze(2)

#         • Identical logic to src_mask, but applied to decoder_input.
#         • Shape: [B, 1, 1, max_tgt_len]
#         • Flags False anywhere a target token is just padding.

# ----------------------------------------------------------------------------------------------------
# 3. CAUSAL / LOOK-AHEAD MASK (causal_mask)
# ----------------------------------------------------------------------------------------------------
#         Line: causal_mask = torch.tril(torch.ones((max_tgt_len, max_tgt_len), dtype=torch.bool)).unsqueeze(0).unsqueeze(1)

#         • The Problem:
#         During training, we give the decoder the whole ground-truth summary at once (teacher-forcing).
#         If position 1 ("Amanda") could look ahead at position 2 ("baked"), training would be trivial—
#         the model would just cheat and copy the next token!

#         • The Solution: Lower Triangular Matrix (torch.tril)
#         Suppose max_tgt_len = 4. torch.ones((4, 4)) creates a full 4x4 grid of 1s.
#         torch.tril(...) keeps only the lower triangle (on and below the main diagonal):

#                             Key:  t0(SOS)  t1(Amanda)  t2(baked)  t3(cookies)
#             Query: t0 (<SOS>)       [[True,     False,      False,     False],   <-- Can only see t0
#             Query: t1 (Amanda)       [True,     True,       False,     False],   <-- Can see t0, t1
#             Query: t2 (baked)        [True,     True,       True,      False],   <-- Can see t0, t1, t2
#             Query: t3 (cookies)      [True,     True,       True,      True ]]   <-- Can see all past

#         • .unsqueeze(0).unsqueeze(1) transforms shape from [max_tgt_len, max_tgt_len] to:
#         Shape: [1, 1, max_tgt_len, max_tgt_len] (adds batch dimension 1 and head dimension 1).

# ----------------------------------------------------------------------------------------------------
# 4. COMBINING BOTH INTO tgt_mask
# ----------------------------------------------------------------------------------------------------
#         Line: tgt_mask = tgt_pad_mask & causal_mask

#         • For decoder self-attention, a token position is valid ONLY if BOTH conditions are met:
#         1. It is NOT in the future (enforced by causal_mask).
#         2. It is NOT a <PAD> token (enforced by tgt_pad_mask).

#         • How broadcasting works:
#         tgt_pad_mask shape: [B, 1, 1,           max_tgt_len]
#         causal_mask  shape: [1, 1, max_tgt_len, max_tgt_len]
#         '&' (bitwise AND) broadcasts them together to yield:
#         Final Shape:        [B, 1, max_tgt_len, max_tgt_len]

#         • Visual Example:
#         If a summary has 2 real tokens and 2 padding tokens: ["<SOS>", "Amanda", "<PAD>", "<PAD>"]

#         tgt_pad_mask = [1, 1, 0, 0]

#         causal_mask  = [[1, 0, 0, 0],
#                         [1, 1, 0, 0],
#                         [1, 1, 1, 0],
#                         [1, 1, 1, 1]]

#         Applying (tgt_pad_mask & causal_mask) zeros out BOTH future tokens AND columns where Key is <PAD>:

#         tgt_mask     = [[1, 0, 0, 0],
#                         [1, 1, 0, 0],
#                         [1, 1, 0, 0],
#                         [1, 1, 0, 0]]

# ----------------------------------------------------------------------------------------------------
# 5. HOW THIS IS CONSUMED IN ATTENTION (model.py):
# ----------------------------------------------------------------------------------------------------
#         Later in your attention layer, this boolean mask is applied directly to the raw attention scores:

#         if mask is not None:
#             scores = scores.masked_fill(mask == False, -1e9)

#         attention_weights = F.softmax(scores, dim=-1)
#         # Any position where mask was False is now softmax(-1e9) ≈ 0.0000!
# ====================================================================================================


In [23]:
# ====================================================================================================
# DEEP DIVE: WHY DO WE NEED ATTENTION MASKS IF LOSS HAS `ignore_index=PAD_IDX`?
# ====================================================================================================
#
# [THE COMMON QUESTION]:
# "In PyTorch, nn.CrossEntropyLoss(ignore_index=PAD_IDX) automatically ignores <PAD> tokens
# when computing loss. So why do we still need 'src_mask', 'tgt_pad_mask', and 'causal_mask'
# inside the DataLoader / Model?"
#
# [THE SHORT ANSWER]:
# The Loss Function and Attention Masks operate at completely different stages of the model:
#   • Attention Masks control: INTERNAL INFORMATION FLOW (How tokens talk to each other inside layers)
#   • ignore_index controls:   EXTERNAL EVALUATION / LOSS (Which output positions get graded)
#
# ----------------------------------------------------------------------------------------------------
# 1. REASON #1: PREVENTING "INFORMATION POLLUTION" (Internal Word Representations)
# ----------------------------------------------------------------------------------------------------
# • In Self-Attention, every token creates its contextual representation by calculating a
#   weighted average of ALL other tokens' Value vectors (V):
#
#       Output(Word) = a1*V(Word_1) + a2*V(Word_2) + ... + a_pad*V(<PAD>)
#
# • If we DO NOT mask <PAD> tokens in attention:
#   1. The Softmax function is forced to allocate positive probability weight (a_pad > 0) to <PAD>.
#   2. Real words (e.g., "Amanda", "cookies") will literally absorb numeric values from meaningless
#      padding positions!
#   3. Their internal embeddings become corrupted by padding noise before reaching higher layers.
#
# • By setting Attention Mask for <PAD> to False (which turns into -1e9 before Softmax):
#   1. Softmax(-1e9) ≈ 0.0000.
#   2. <PAD> tokens receive EXACTLY 0% attention weight.
#   3. Real words ONLY exchange meaning with other REAL words.
#
# ----------------------------------------------------------------------------------------------------
# 2. REASON #2: PREVENTING "CHEATING" (The Causal / Look-Ahead Mask)
# ----------------------------------------------------------------------------------------------------
# • During training, we feed the entire target summary at once into the decoder (Teacher Forcing).
# • When the model is predicting word #2 ("baked"), it CANNOT be allowed to look at word #2 or #3!
# • 'ignore_index' in CrossEntropyLoss has ZERO awareness of past vs. future. It is just an error
#   calculator at the very end of the network.
# • Without a CAUSAL MASK inside the decoder's Self-Attention:
#   1. The decoder would simply look ahead into future positions, cheat, and copy the next token.
#   2. Training loss would drop to zero immediately, but during inference (when future words
#      do not exist yet), the model would completely fail and output garbage.
#
# ----------------------------------------------------------------------------------------------------
# 3. SUMMARY: WHO DOES WHAT?
# ----------------------------------------------------------------------------------------------------
# +--------------------+----------------------------+------------------------------------------------+
# | Mechanism          | Location                   | Responsibility                                 |
# +--------------------+----------------------------+------------------------------------------------+
# | Attention Masks    | INSIDE the Transformer     | 1. Stops real words from absorbing <PAD> noise.|
# | (src_mask, tgt_mask| Layers (Before Softmax)    | 2. Stops decoder from peeking at future tokens.|
# +--------------------+----------------------------+------------------------------------------------+
# | ignore_index=0     | AT THE END of the network  | Tells PyTorch: "Don't calculate penalty/loss   |
# |                    | in CrossEntropyLoss        | for predictions made at dummy <PAD> positions".|
# +--------------------+----------------------------+------------------------------------------------+
#
# ----------------------------------------------------------------------------------------------------
# 4. TENSOR SHAPE & BROADCASTING EXPLANATION:
# ----------------------------------------------------------------------------------------------------
# Raw Attention Scores Tensor: [Batch_Size, Num_Heads, Query_Len, Key_Len]
#
# • src_mask:
#   (encoder_input != pad_idx)                       -> [B, max_src_len]
#   .unsqueeze(1)                                    -> [B, 1, max_src_len]           (for Num_Heads)
#   .unsqueeze(2)                                    -> [B, 1, 1, max_src_len]        (for Query_Len)
#   Broadcasts across all heads and queries to block <PAD> Keys!
#
# • causal_mask:
#   torch.tril(torch.ones(max_tgt_len, max_tgt_len)) -> [max_tgt_len, max_tgt_len]   (Lower Triangle)
#   .unsqueeze(0).unsqueeze(1)                       -> [1, 1, max_tgt_len, max_tgt_len]
#
# • tgt_mask = tgt_pad_mask & causal_mask:
#   Bitwise AND combines both: token must be NOT in future AND NOT a <PAD> token!
#   Broadcasts to shape: [B, 1, max_tgt_len, max_tgt_len]
# ====================================================================================================

##### Continue Code

In [24]:
# =====================================================================
# Instantiate Datasets & DataLoaders
# =====================================================================
BATCH_SIZE = 16
MAX_SRC_LEN = 128
MAX_TGT_LEN = 64

In [25]:
# 1. Create Datasets
train_dataset = data_loader(train_df, tokenizer, max_src_len=MAX_SRC_LEN, max_tgt_len=MAX_TGT_LEN)
val_dataset   = data_loader(val_df, tokenizer, max_src_len=MAX_SRC_LEN, max_tgt_len=MAX_TGT_LEN)
test_dataset  = data_loader(test_df, tokenizer, max_src_len=MAX_SRC_LEN, max_tgt_len=MAX_TGT_LEN)

train_dataset[0]

{'encoder_input': [2064,
  29,
  44,
  7872,
  4459,
  17,
  1473,
  973,
  1108,
  1099,
  34,
  2502,
  29,
  1550,
  4,
  2064,
  29,
  44,
  10,
  1069,
  1584,
  973,
  1253,
  2280],
 'decoder_input': [2, 2064, 7872, 4459, 990, 1019, 1584, 2502, 1099, 1253, 17],
 'decoder_target': [2064,
  7872,
  4459,
  990,
  1019,
  1584,
  2502,
  1099,
  1253,
  17,
  3]}

In [26]:
# 2. Create DataLoaders

train_loader = DataLoader(  train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                            collate_fn=lambda b: data_loader.collate_fn(b, pad_idx=PAD_IDX),
                            pin_memory=True if torch.cuda.is_available() else False)

val_loader = DataLoader(    val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                            collate_fn=lambda b: data_loader.collate_fn(b, pad_idx=PAD_IDX),
                            pin_memory=True if torch.cuda.is_available() else False)

test_loader = DataLoader(   test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                            collate_fn=lambda b: data_loader.collate_fn(b, pad_idx=PAD_IDX),
                            pin_memory=True if torch.cuda.is_available() else False)

print(f"Total Batches -> Train : {len(train_loader):,} \n\t\t Val   : {len(val_loader):,} \n\t\t Test  : {len(test_loader):,}\n")

sample_batch = next(iter(train_loader))
encoder_input  = sample_batch["encoder_input"]   # Shape: [B, max_src_len]
decoder_input  = sample_batch["decoder_input"]   # Shape: [B, max_tgt_len]
decoder_target = sample_batch["decoder_target"]  # Shape: [B, max_tgt_len]
src_mask       = sample_batch["src_mask"]        # Shape: [B, 1, 1, max_src_len]
tgt_mask       = sample_batch["tgt_mask"]        # Shape: [B, 1, max_tgt_len, max_tgt_len]

print(f"Batch Size             : {encoder_input.size(0)}")
print(f"encoder_input shape    : {encoder_input.shape}")
print(f"decoder_input shape    : {decoder_input.shape}")

Total Batches -> Train : 918 
		 Val   : 52 
		 Test  : 51

Batch Size             : 16
encoder_input shape    : torch.Size([16, 128])
decoder_input shape    : torch.Size([16, 34])


In [27]:
# Check Teacher-Forcing Alignment for 1st item in batch
sample_dec_in  = sample_batch['decoder_input'][0].tolist()
sample_dec_tgt = sample_batch['decoder_target'][0].tolist()

print("Teacher-Forcing Shift Check (first 6 tokens):")
print(f"  decoder_input  : {sample_dec_in[:7]}")
print(f"  decoder_target : {sample_dec_tgt[:6]}")

Teacher-Forcing Shift Check (first 6 tokens):
  decoder_input  : [2, 4819, 1126, 586, 2056, 967, 5903]
  decoder_target : [4819, 1126, 586, 2056, 967, 5903]


# 7) Model

In [28]:
# ====================================================================================================
# ARCHITECTURAL BLUEPRINT: ENCODER-DECODER TRANSFORMER FROM SCRATCH
# ====================================================================================================
#
# ----------------------------------------------------------------------------------------------------
# 1. COMPLETE SYSTEM FLOWCHART
# ----------------------------------------------------------------------------------------------------
#
#      SOURCE DIALOGUE (Article)                        TARGET SUMMARY (Shifted Context)
#      Shape: [Batch, max_src_len]                      Shape: [Batch, max_tgt_len]
#                 │                                                 │
#                 ▼                                                 ▼
#       Token Embedding Layer                             Token Embedding Layer
#                 │                                                 │
#                 ▼                                                 ▼
#       + Positional Encoding                             + Positional Encoding
#                 │                                                 │
#                 ▼                                                 ▼
#     ┌────────────────────────┐                        ┌────────────────────────┐
#     │     ENCODER STACK      │                        │     DECODER STACK      │
#     │  (N identical layers)  │                        │  (N identical layers)  │
#     │                        │                        │                        │
#     │  1. Self-Attention     │                        │  1. Masked Causal      │
#     │     + Add & Norm       │                        │     Self-Attention     │
#     │                        │                        │     + Add & Norm       │
#     │  2. Feed-Forward       │                        │                        │
#     │     + Add & Norm       │               ┌───────►│  2. Cross-Attention    │
#     │                        │               │        │     (Q: Dec, K,V: Enc) │
#     └───────────┬────────────┘               │        │     + Add & Norm       │
#                 │                            │        │                        │
#                 │ Context Keys & Values      │        │  3. Feed-Forward       │
#                 └────────────────────────────┘        │     + Add & Norm       │
#                                                       └───────────┬────────────┘
#                                                                   │
#                                                                   ▼
#                                                         Linear Output Head
#                                                         (d_model -> vocab_size)
#                                                                   │
#                                                                   ▼
#                                                             OUTPUT LOGITS
#                                                     [Batch, max_tgt_len, vocab_size]
#
# ----------------------------------------------------------------------------------------------------
# 2. THE 7 CORE BUILDING BLOCKS (BOTTOM-UP IMPLEMENTATION)
# ----------------------------------------------------------------------------------------------------
#
#  [1] PositionalEncoding (nn.Module)
#      • Purpose: Transformers process all tokens simultaneously with zero inherent sense of order.
#      • Formula: Sinusoidal waves of varying frequencies:
#                 PE(pos, 2i)   = sin(pos / 10000^(2i / d_model))
#                 PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))
#      • Input:   [Batch, Seq_Len, d_model]
#      • Output:  [Batch, Seq_Len, d_model]
#
#  [2] MultiHeadAttention (nn.Module)
#      • Purpose: Allows each token to attend to multiple representation subspaces simultaneously.
#      • Single reusable module used for all 3 attention types in the Transformer:
#          - Encoder Self-Attention:       Q = Enc_In,  K = Enc_In,  V = Enc_In  (uses src_mask)
#          - Decoder Causal Self-Attention: Q = Dec_In,  K = Dec_In,  V = Dec_In  (uses tgt_mask)
#          - Decoder Cross-Attention:       Q = Dec_Mid, K = Enc_Out, V = Enc_Out (uses src_mask)
#      • Scaled Dot-Product Formula:
#                 Attention(Q, K, V) = Softmax( (Q @ K.T) / sqrt(d_k) + Mask ) @ V
#      • Shape:   [Batch, Heads, Q_Len, d_k] x [Batch, Heads, d_k, K_Len] -> [Batch, Heads, Q_Len, K_Len]
#
#  [3] PositionwiseFeedForward (nn.Module)
#      • Purpose: 2-layer MLP applied identically and independently to each token position.
#      • Formula: FFN(x) = max(0, x @ W1 + b1) @ W2 + b2   (Linear -> ReLU/GELU -> Dropout -> Linear)
#      • Projections: d_model (256) -> d_ff (512/1024) -> d_model (256)
#
#  [4] EncoderLayer (nn.Module)
#      • Sublayer 1: Multi-Head Self-Attention + Residual Add & Layer Normalization
#      • Sublayer 2: Position-wise Feed-Forward  + Residual Add & Layer Normalization
#
#  [5] DecoderLayer (nn.Module)
#      • Sublayer 1: Masked Causal Multi-Head Self-Attention + Add & LayerNorm
#      • Sublayer 2: Encoder-Decoder Cross-Attention          + Add & LayerNorm
#      • Sublayer 3: Position-wise Feed-Forward               + Add & LayerNorm
#
#  [6] Encoder & Decoder Stacks (nn.Module)
#      • Encoder: Embedding Layer (scaled by sqrt(d_model)) -> PositionalEncoding -> N x EncoderLayer -> LayerNorm
#      • Decoder: Embedding Layer (scaled by sqrt(d_model)) -> PositionalEncoding -> N x DecoderLayer -> LayerNorm
#
#  [7] Seq2SeqTransformer (nn.Module)
#      • Encompasses Encoder stack, Decoder stack, and final Linear projection layer to target vocab.
#      • Returns raw vocabulary logits: [Batch, max_tgt_len, VOCAB_SIZE].
#
# ----------------------------------------------------------------------------------------------------
# 3. TENSOR DIMENSIONS FLOW (BASED ON OUR CURRENT BATCH CONFIGURATION)
# ----------------------------------------------------------------------------------------------------
#  +--------------------------+------------------------------------+---------------------------------+
#  | Processing Stage         | Input Tensors & Shapes             | Output Tensor Shape             |
#  +--------------------------+------------------------------------+---------------------------------+
#  | 1. Source Embedding + PE | encoder_input: [16, 128]           | [16, 128, 256]                  |
#  | 2. Encoder Stack (x3)    | [16, 128, 256], src_mask           | enc_out: [16, 128, 256]         |
#  | 3. Target Embedding + PE | decoder_input: [16, 44]            | [16, 44, 256]                   |
#  | 4. Decoder Stack (x3)    | [16, 44, 256], enc_out, tgt_mask   | dec_out: [16, 44, 256]          |
#  | 5. Output Projection     | dec_out: [16, 44, 256]             | logits: [16, 44, 8000]          |
#  +--------------------------+------------------------------------+---------------------------------+
# ====================================================================================================

In [31]:
# ====================================================================================================
# 1. POSITIONAL ENCODING CLASS
# ====================================================================================================

class PositionalEncoding(nn.Module):
    """
    Implements the standard Sinusoidal Positional Encoding from
    'Attention Is All You Need' (Vaswani et al., 2017).

    Since self-attention processes all tokens simultaneously with permutation
    invariance, we inject geometric positional signals into the token embeddings:

        PE(pos, 2i)   = sin( pos / 10000^(2i / d_model) )
        PE(pos, 2i+1) = cos( pos / 10000^(2i / d_model) )

    Args:
        d_model (int): Hidden dimension size of token embeddings (e.g., 256).
        dropout (float): Dropout probability applied after adding PE to embeddings.
        max_len (int): Maximum sequence length to precompute positional encodings for.
    """

    def __init__(self, 
                d_model : int, 
                dropout : float = 0.1, 
                max_len : int = 5000 ):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # 1. Allocate PE matrix of shape [max_len, d_model]
        pe = torch.zeros(max_len, d_model)

        # 2. Position indices column vector: shape [max_len, 1]
        #    e.g., [[0.], [1.], [2.], ..., [max_len-1.]]
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 3. Frequency scaling term (div_term) for even dimensions (2i):
        #    10000^(2i / d_model) = exp( 2i * -log(10000.0) / d_model )
        #    Computes in log-space for superior numerical stability.
        #    Shape: [d_model // 2]
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # 4. Fill even embedding dimensions (0, 2, 4, ...) with sine waves:
        pe[:, 0::2] = torch.sin(position * div_term)

        # 5. Fill odd embedding dimensions (1, 3, 5, ...) with cosine waves:
        pe[:, 1::2] = torch.cos(position * div_term)

        # 6. Add a batch dimension: shape [1, max_len, d_model]
        #    This allows seamless broadcasting across any batch size [B, seq_len, d_model]
        pe = pe.unsqueeze(0)

        # 7. Register as buffer:
        #    - Saved into state_dict
        #    - Automatically moved to GPU/CPU with model.to(device)
        #    - NOT updated by optimizer gradients (requires_grad = False)
        #    - This code automatically binds pe variable to self.pe variable 
        #    - We do it this way so that in future we can automatically move devices
        #    - This freezes the pe variable with the class so when the class moves devices, this variable moves together 
        #    - Otherwise we would have to manually do it outselves, can cause errors ..... This is safer
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Token embeddings tensor of shape [Batch_Size, Seq_Len, d_model].

        Returns:
            torch.Tensor: Position-infused embeddings with dropout, shape [Batch_Size, Seq_Len, d_model].
        """
        # Slice precomputed PE up to the sequence length of the current batch (x.size(1))
        # pe[:, :x.size(1)] has shape [1, Seq_Len, d_model] and broadcasts across Batch_Size
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)



# ====================================================================================================
# SANITY & SHAPE VERIFICATION TEST
# ====================================================================================================

d_model = 256
batch_size = 16
seq_len = 128

# Create dummy token embeddings
dummy_embeddings = torch.zeros(batch_size, seq_len, d_model)

# Initialize PositionalEncoding
pos_encoder = PositionalEncoding(d_model=d_model, dropout=0.1, max_len=512)
output = pos_encoder(dummy_embeddings)

print("PositionalEncoding initialized successfully!")
print(f"  • Input Shape  : {dummy_embeddings.shape}")
print(f"  • Output Shape : {output.shape}")
print(f"  • Output device: {output.device}")
assert output.shape == (batch_size, seq_len, d_model), "Shape mismatch in PositionalEncoding!"

PositionalEncoding initialized successfully!
  • Input Shape  : torch.Size([16, 128, 256])
  • Output Shape : torch.Size([16, 128, 256])
  • Output device: cpu


In [ ]:
# ====================================================================================================
# 2. MULTI-HEAD ATTENTION CLASS
# 
# REFERENCE CONFIGURATION & SHAPES -
#
# Model Hyperparameters:
#   • d_model   = 256           (Embedding / hidden state dimension)
#   • num_heads = 4             (Number of parallel attention heads)
#   • d_k       = 256 // 4 = 64 (Dimension per attention head)
#
# Dynamic Batch Dimensions (from DataLoader dynamic padding collate_fn):
#   • Batch Size: <batch_size> = 16
#   • Query Len : <q_len>      = Dynamic (e.g., 44 for decoder or 128 for encoder)
#   • Key/Val Len: <k_len>     = Dynamic (e.g., 128 for encoder or 44 for decoder)
# ====================================================================================================

class MultiHeadAttention(nn.Module):

    """
    Implements Multi-Head Attention as described in 'Attention Is All You Need'.

    Allows the model to jointly attend to information from different representation
    subspaces at different positions:
        MultiHead(Q, K, V) = Concat(head_1, ..., head_h) @ W_o
        head_i = Attention(Q @ W_q_i, K @ W_k_i, V @ W_v_i)

    Args:
        d_model (int): Total dimensionality of the model (e.g., 256).
        num_heads (int): Number of parallel attention heads (e.g., 4 or 8).
        dropout (float): Dropout probability applied to attention weights.
    """

    def __init__(self, 
                d_model : int, 
                num_heads : int, 
                dropout : float = 0.1):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, f"d_model ({d_model}) must be divisible by num_heads ({num_heads})"
        
        self.d_model = d_model              # self.d_model   = 256
        self.num_heads = num_heads          # self.num_heads = 4
        self.d_k = d_model // num_heads     # self.d_k       = 64
        
        # --------------------------------------------------------------------------------------------
        # Linear Projections (Maps d_model -> d_model before splitting into heads)
        # Weight Shape: [out_features, in_features] = [256, 256] | Bias Shape: [256]
        # --------------------------------------------------------------------------------------------

        self.w_q = nn.Linear(d_model, d_model)      # W_q: (256, 256)
        self.w_k = nn.Linear(d_model, d_model)      # W_k: (256, 256)
        self.w_v = nn.Linear(d_model, d_model)      # W_v: (256, 256)
        
        # Final output projection after concatenating heads
        self.fc_out = nn.Linear(d_model, d_model)   # fc_out: (256, 256)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, 
                q: torch.Tensor, 
                k: torch.Tensor, 
                v: torch.Tensor, 
                mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Input Tensor Shapes:
          • q:    [<batch_size>, <q_len>, 256]
          • k:    [<batch_size>, <k_len>, 256]
          • v:    [<batch_size>, <k_len>, 256]
          • mask: [<batch_size>, 1, 1, <k_len>]        (for source padding mask) OR
                  [<batch_size>, 1, <q_len>, <k_len>]  (for target causal mask)
        """

        # ============================================================================================
        # UNDERSTANDING Q, K, V SOURCES AND SHAPES:
        #
        # 1. ENCODER SELF-ATTENTION:
        #    • Q, K, V all come from the source article: q = k = v = encoder_input [16, 128, 256]
        #    • Here: q_len = 128, k_len = 128, v_len = 128 (all equal)
        #
        # 2. DECODER CAUSAL SELF-ATTENTION:
        #    • Q, K, V all come from the target summary: q = k = v = decoder_input [16, 44, 256]
        #    • Here: q_len = 44,  k_len = 44,  v_len = 44  (all equal)
        #
        # 3. DECODER CROSS-ATTENTION (Why q_len and k_len can be different!):
        #    • Q comes from the summary (Decoder): q = decoder_hidden_state   [16, 44, 256]
        #    • K & V come from the article (Encoder): k = v = encoder_output   [16, 128, 256]
        #    • Here: q_len = 44 (summary tokens looking for info)
        #            k_len = 128, v_len = 128 (article tokens providing info)
        # ============================================================================================

        batch_size = q.size(0)  # batch_size = <batch_size> (e.g. 16)
        q_len = q.size(1)       # q_len      = <q_len>      (e.g. 44)
        k_len = k.size(1)       # k_len      = <k_len>      (e.g. 128)
        v_len = v.size(1)       # v_len      = <k_len>      (e.g. 128)

        # --------------------------------------------------------------------------------------------
        # STEP 1: Linear Projections & Reshaping into Heads
        # --------------------------------------------------------------------------------------------
        # self.w_q(q)                                      -> Shape: [<batch_size>, <q_len>, d_model] 
        #                                                  ->        [<batch_size>, <q_len>, 256]
        #       self.w_q = nn.Linear(d_model, d_model) means it is a linear layer function 
        #           It takes in a vector of (N, d_model) 
        #           Passes through a FC layer of (d_model, d_model) and adds a bias of (d_model = 256) 
        #           Basically nn.Linear(d_model, d_model) is a full function of FC layer, not only a matrix -> (Wx + B)
        # .view(batch_size, q_len, num_heads, d_k)         -> Shape: [<batch_size>, <q_len>, 4, 64]
        # .transpose(1, 2)                                 -> Shape: [<batch_size>, 4, <q_len>, 64]

        # self.w_k(k)                                      -> Shape: [<batch_size>, <k_len>, d_model] 
        #                                                  ->        [<batch_size>, <k_len>, 256]
        # .view(batch_size, k_len, num_heads, d_k)         -> Shape: [<batch_size>, <k_len>, 4, 64]
        # .transpose(1, 2)                                 -> Shape: [<batch_size>, 4, <k_len>, 64]
        
        # self.w_v(v)                                      -> Shape: [<batch_size>, <k_len>, 256]
        #                                                  ->        [<batch_size>, <k_len>, 256]
        # .view(batch_size, v_len, num_heads, d_k)         -> Shape: [<batch_size>, <k_len>, 4, 64]
        # .transpose(1, 2)                                 -> Shape: [<batch_size>, 4, <k_len>, 64]

        Q = self.w_q(q).view(batch_size, q_len, self.num_heads, self.d_k).transpose(1, 2)   # Q  : [<batch_size>, num_heads, <q_len>, d_k] 
                                                                                            #   -> [<batch_size>, 4, <q_len>, 64]
        K = self.w_k(k).view(batch_size, k_len, self.num_heads, self.d_k).transpose(1, 2)   # K  : [<batch_size>, num_heads, <k_len>, d_k] 
                                                                                            #   -> [<batch_size>, 4, <k_len>, 64]
        V = self.w_v(v).view(batch_size, v_len, self.num_heads, self.d_k).transpose(1, 2)   # V  : [<batch_size>, num_heads, <k_len>, d_k] 
                                                                                            #   -> [<batch_size>, 4, <k_len>, 64]
        
        # --------------------------------------------------------------------------------------------
        # STEP 2: Scaled Dot-Product Attention Scores
        # Q                     : [<batch_size>, num_heads, <q_len>, d_k] -> [<batch_size>, 4, <q_len>, 64]
        # K.transpose(-2, -1)   : [<batch_size>, num_heads, d_k, <k_len>] -> [<batch_size>, 4, 64, <k_len>]
        #       K.transpose(-2, -1) flips the last two dimensions: [B, H, k_len, d_k] -> [B, H, d_k, k_len]
        #       Example: [16, 4, 128, 64] becomes [16, 4, 64, 128]
        #       This matches inner dimensions so : Q [16, 4, 44, 64] @ K^T [16, 4, 64, 128] = [16, 4, 44, 128]
        # Matrix Multiplication :   [<batch_size>, num_heads, <q_len>, d_k] @ [<batch_size>, num_heads, d_k, <k_len>]
        #                        -> [<batch_size>, 4, <q_len>, 64] @ [<batch_size>, 4, 64, <k_len>] -> [<batch_size>, 4, <q_len>, <k_len>]
        # --------------------------------------------------------------------------------------------
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)     # scores: [<batch_size>, num_heads, <q_len>, <k_len>]
                                                                                # scores: [<batch_size>, 4, <q_len>, <k_len>]

        # --------------------------------------------------------------------------------------------
        # STEP 3: Apply Mask (Broadcasts across heads and query positions)
        # mask shape:   [<batch_size>, 1, 1, <k_len>] OR [<batch_size>, 1, <q_len>, <k_len>]
        # scores shape: [<batch_size>, num_heads, <q_len>, <k_len>] -> [<batch_size>, 4, <q_len>, <k_len>]
        # --------------------------------------------------------------------------------------------
        if mask is not None:
            scores = scores.masked_fill(mask == False, -1e9)                # scores: [<batch_size>, num_heads, <q_len>, <k_len>]
                                                                            #      -> [<batch_size>, 4, <q_len>, <k_len>]

        # --------------------------------------------------------------------------------------------
        # STEP 4: Softmax & Dropout (Normalizes across the last dimension: k_len)
        # --------------------------------------------------------------------------------------------
        attn_weights = torch.softmax(scores, dim=-1)                        # attn_weights: [<batch_size>, num_heads, <q_len>, <k_len>]
                                                                            #            -> [<batch_size>, 4, <q_len>, <k_len>]
        attn_weights = self.dropout(attn_weights)                           # attn_weights: [<batch_size>, num_heads, <q_len>, <k_len>]
                                                                            #            -> [<batch_size>, 4, <q_len>, <k_len>]

        # --------------------------------------------------------------------------------------------
        # STEP 5: Multiply Attention Weights by Value vectors
        # attn_weights : [<batch_size>, num_heads, <q_len>, <k_len>] -> [<batch_size>, 4, <q_len>, <k_len>]
        # V            : [<batch_size>, num_heads, <k_len>, d_k] -> [<batch_size>, 4, <k_len>, 64]
        # Matrix Mult  : [<batch_size>, num_heads, <q_len>, <k_len>] @ [<batch_size>, num_heads, <k_len>, d_k] 
        #                [<batch_size>, 4, <q_len>, <k_len>] @ [<batch_size>, 4, <k_len>, 64] -> [<batch_size>, 4, <q_len>, 64]
        # --------------------------------------------------------------------------------------------
        context = torch.matmul(attn_weights, V)                             # context: [<batch_size>, num_heads, <q_len>, d_k]
                                                                            #          [<batch_size>, 4, <q_len>, 64]

        # --------------------------------------------------------------------------------------------
        # STEP 6: Concatenate All Heads Together
        # context.transpose(1, 2)                           -> Shape: [<batch_size>, <q_len>, num_heads, d_k] 
        #                                                   ->        [<batch_size>, <q_len>, 4, 64] 
        # .contiguous().view(batch_size, q_len, d_model)    -> Shape: [<batch_size>, <q_len>, 256]
        # --------------------------------------------------------------------------------------------
        context = context.transpose(1, 2).contiguous().view(batch_size, q_len, self.d_model) # context: [<batch_size>, <q_len>, 256]
        
        # --------------------------------------------------------------------------------------------
        # STEP 7: Final Linear Projection (d_model)
        # self.fc_out: maps input dimension 256 -> 256
        # --------------------------------------------------------------------------------------------
        output = self.fc_out(context)                                       # output: [<batch_size>, <q_len>, d_model]
                                                                            #         [<batch_size>, <q_len>, 256]
        return output



# ====================================================================================================
# SHAPE VERIFICATION TEST
# ====================================================================================================

batch_size = 16
d_model = 256
num_heads = 4
q_len = 44   # e.g. target sequence length
k_len = 128  # e.g. source sequence length

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads, dropout=0.1)

# 1. Test Self-Attention: Q, K, V all have length = q_len (44)
x = torch.randn(batch_size, q_len, d_model)                          # x: [16, 44, 256]
self_attn_out = mha(q=x, k=x, v=x, mask=None)                        # self_attn_out: [16, 44, 256]
print("1. Self-Attention Output Shape  :", self_attn_out.shape)
assert self_attn_out.shape == (batch_size, q_len, d_model)

# 2. Test Cross-Attention: Q has length 44, K and V have length 128 (enc_out)
enc_out = torch.randn(batch_size, k_len, d_model)                        # enc_out: [16, 128, 256]
dummy_src_mask = torch.ones((batch_size, 1, 1, k_len), dtype=torch.bool) # mask: [16, 1, 1, 128]
cross_attn_out = mha(q=x, k=enc_out, v=enc_out, mask=dummy_src_mask)     # cross_attn_out: [16, 44, 256]
print("2. Cross-Attention Output Shape :", cross_attn_out.shape)
assert cross_attn_out.shape == (batch_size, q_len, d_model)

1. Self-Attention Output Shape  : torch.Size([16, 44, 256])
2. Cross-Attention Output Shape : torch.Size([16, 44, 256])


In [38]:
# ====================================================================================================
# 3. PositionWiseFeedForward CLASS
# 
# REFERENCE CONFIGURATION & SHAPES FOR COMMENTS BELOW:
#
# Model Hyperparameters:
#   • d_model = 256           (Embedding / hidden state dimension)
#   • d_ff    = 1024          (Inner feed-forward expansion dimension, typically 2x-4x of d_model)
#
# Dynamic Batch Dimensions (from DataLoader dynamic padding collate_fn):
#   • Batch Size     : <batch_size> = 16
#   • Sequence Length: <seq_len>    = Dynamic (e.g., 128 for encoder or 44 for decoder)
# ====================================================================================================

class PositionWiseFeedForward(nn.Module):

    """
    A two-layer fully connected MLP applied identically and independently to each token position:
        FFN(x) = max(0, x @ W1 + b1) @ W2 + b2

    Args:
        d_model (int): Model hidden dimension (e.g., 256).
        d_ff (int): Hidden dimension of the intermediate expansion layer (e.g., 512 or 1024).
        dropout (float): Dropout probability applied after the activation function.
    """

    def __init__(self, 
                d_model : int, 
                d_ff : int = 1024, 
                dropout : float = 0.1):
        super(PositionWiseFeedForward, self).__init__()
        
        self.d_model = d_model              # self.d_model = 256
        self.d_ff = d_ff                    # self.d_ff    = 1024
        
        # --------------------------------------------------------------------------------------------
        # Layer 1: Projects from d_model up to intermediate expanded space d_ff
        # Weight Shape: [out_features, in_features] = [1024, 256] | Bias Shape: [1024]
        # --------------------------------------------------------------------------------------------
        self.linear1 = nn.Linear(d_model, d_ff)         # W_1: (1024, 256)
        
        # Non-linear activation function
        self.activation = nn.ReLU()
        
        # Dropout applied to activated intermediate representations
        self.dropout = nn.Dropout(p=dropout)
        
        # --------------------------------------------------------------------------------------------
        # Layer 2: Projects back down from d_ff to d_model
        # Weight Shape: [out_features, in_features] = [256, 1024] | Bias Shape: [256]
        # --------------------------------------------------------------------------------------------
        self.linear2 = nn.Linear(d_ff, d_model)         # W_2: (256, 1024)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Input Tensor Shape:
          • x: [<batch_size>, <seq_len>, 256]

        Returns:
          • Output Tensor Shape: [<batch_size>, <seq_len>, 256]
        """
        # --------------------------------------------------------------------------------------------
        # STEP 1: First Linear Projection (Expands from d_model=256 -> d_ff=1024)
        # x shape: [<batch_size>, <seq_len>, 256]
        # self.linear1: maps 256 -> 1024
        #              [<batch_size>, <seq_len>, 256] @ [256, 1024] -> [<batch_size>, <seq_len>, 1024]
        # --------------------------------------------------------------------------------------------
        out = self.linear1(x)                                   # out: [<batch_size>, <seq_len>, 1024]

        # --------------------------------------------------------------------------------------------
        # STEP 2: Non-Linear Activation (ReLU)
        # Replaces all negative values with 0
        # --------------------------------------------------------------------------------------------
        out = self.activation(out)                              # out: [<batch_size>, <seq_len>, 1024]

        # --------------------------------------------------------------------------------------------
        # STEP 3: Dropout Regularization
        # Randomly zeroes out some elements with probability 'dropout' during training
        # --------------------------------------------------------------------------------------------
        out = self.dropout(out)                                 # out: [<batch_size>, <seq_len>, 1024]

        # --------------------------------------------------------------------------------------------
        # STEP 4: Second Linear Projection (Projects from d_ff=1024 back to d_model=256)
        # self.linear2: maps 1024 -> 256
        #              [<batch_size>, <seq_len>, 1024] @ [1024, 256] -> [<batch_size>, <seq_len>, 256]
        # --------------------------------------------------------------------------------------------
        out = self.linear2(out)                                  # out: [<batch_size>, <seq_len>, 256]

        return out


# ====================================================================================================
# SANITY & SHAPE VERIFICATION TEST
# ====================================================================================================

batch_size = 16
seq_len = 128  # e.g., article sequence length
d_model = 256
d_ff = 1024

ffn = PositionWiseFeedForward(d_model=d_model, d_ff=d_ff, dropout=0.1)

# Create dummy input tensor
dummy_x = torch.randn(batch_size, seq_len, d_model)             # dummy_x: [16, 128, 256]
ffn_out = ffn(dummy_x)                                          # ffn_out: [16, 128, 256]

print(f"  • Input Shape  : {dummy_x.shape}")
print(f"  • Output Shape : {ffn_out.shape}")
assert ffn_out.shape == (batch_size, seq_len, d_model), "Feed-forward output shape mismatch!"

  • Input Shape  : torch.Size([16, 128, 256])
  • Output Shape : torch.Size([16, 128, 256])


# 8) Training

# 9) Validation

# 10) Outputs